In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import make_scorer, mean_absolute_error, root_mean_squared_error, r2_score

p22 = pd.read_csv('./data/processed_22.csv')
p23 = pd.read_csv('./data/processed_23.csv')
p24 = pd.read_csv('./data/processed_24.csv')
p25 = pd.read_csv('./data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

a = [ "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
"MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
"Solidity", "texture_uniformity", "texture_smoothness",
"texture_average_gray_level", "texture_entropy",
"texture_average_contrast", "H90",
"H180", "Hflip", "Extent", "EquivDiameter",
"ConvexArea", "ConvexPerimeter" ]

fs = combined[a]

Total days in range: 1184
Total days with data: 805
Days with true missing data: 379


In [78]:
# looking at missing data
# missing data not great for time series analysis, but we can at least identify the gaps
w = pd.to_datetime(
    combined['date'].astype(str),
    format='%Y%m%d'
)

full_dates = pd.date_range(
    start=w.min(),
    end=w.max(),
    freq='D'
)
observed_dates = w
missing_dates = full_dates.difference(observed_dates)

print("Total days in range:", len(full_dates))
print("Total days with data:", len(observed_dates))
print("Days with true missing data:", len(missing_dates))

missing_df = pd.DataFrame({'date': missing_dates})
missing_df['year_month'] = missing_df['date'].dt.to_period('M')
# missing_df['year_month'].value_counts().sort_index()

Total days in range: 1184
Total days with data: 805
Days with true missing data: 379


In [ ]:
env = pd.read_csv('../data/environment_all.csv')
env = env.drop(columns=['fluorescent_dissolved_organic_matter_eco', 'sea_water_turbidity_eco', 'waterlevel_predicted_m'])

# ones of contention (due to too many missing values):
    # mass_concentration_of_oxygen_in_sea_water_seaphox
    # mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox
    # fractional_saturation_of_oxygen_in_sea_water_seaphox
    # sea_water_ph_reported_on_total_scale_seaphox_external

env = env.drop(columns=['mass_concentration_of_oxygen_in_sea_water_seaphox', 'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox', 'fractional_saturation_of_oxygen_in_sea_water_seaphox', 'sea_water_ph_reported_on_total_scale_seaphox_external'])
env.head()

,date,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,20221001,28.717,47.012,33.406,23.250,3.475,21.054,2.066667,216.458333,3.166667,19.250000,1010.879167,0.955167
1,20221002,37.439,46.926,33.418,23.294,3.454,20.957,2.287500,217.166667,3.050000,19.841667,1012.595833,0.927333
2,20221003,46.564,45.266,33.377,23.694,3.456,19.307,2.279167,235.750000,3.087500,20.229167,1014.079167,0.899667
3,20221004,55.428,43.889,33.317,23.990,3.439,17.962,1.491667,172.500000,2.050000,19.504167,1013.445833,0.903083
4,20221005,62.660,44.128,33.304,23.917,3.427,18.227,1.645833,238.833333,2.045833,18.700000,1013.991667,0.892917


In [83]:
y = fs['Lpoly_expected_ml'].copy()
# X = fs.drop(columns=['date','Lpoly_expected_ml', 'roiCount', 'ml_analyzed', 'ROI_per_ml', 'Lpoly_expected', 'Pmicans_expected', 'Pmicans_expected_ml'])
X = fs.drop(columns=['Lpoly_expected_ml'])
y_log = np.log1p(y)

lag = 7
X_lag = X.iloc[:-lag].reset_index(drop=True)
y_lag = y_log.iloc[lag:].reset_index(drop=True)

scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(root_mean_squared_error, greater_is_better=False),
    "r2": make_scorer(r2_score)
}

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    random_state=67,
    n_jobs=-1
)

tscv = TimeSeriesSplit(n_splits=10)

cv_results = cross_validate(
    rf,
    X_lag,
    y_lag,
    cv=tscv,
    scoring=scoring,
    return_train_score=False
)

print("MAE (log): ", -cv_results["test_mae"].mean(), "+/-", cv_results["test_mae"].std())
print("RMSE (log):", -cv_results["test_rmse"].mean(), "+/-", cv_results["test_rmse"].std())
print("R2:",  cv_results["test_r2"].mean(), "+/-", cv_results["test_r2"].std())

MAE (log):  0.5776419426745376 +/- 0.3073913807544037
RMSE (log): 0.7746641174187887 +/- 0.4306123251380448
R2: -66.36800354071666 +/- 129.56537316089378


of a persistent model

Trivial RMSE: 0.42803965205978456

Trivial Correlation: 0.8777423179196997